[Énoncés : Série 7 — Mathématiques et algorithmes gloutons](../serie7_maths_gloutons.ipynb)

# Corrigé — Série 7 — Mathématiques et algorithmes gloutons

Pour chaque exercice : l'idée, la complexité, puis une solution commentée avec ses tests. Il existe souvent plusieurs bonnes solutions : si la tienne passe tous les tests et le test de vitesse, elle est juste. Compare surtout l'**idée**.

Exécute d'abord la cellule d'outils.

In [ ]:
# Outils de ce notebook : exécute cette cellule en premier.
# - verifier(fonction, cas) lance ta fonction sur chaque cas et affiche ✅ (bon), ❌ (mauvais) ou 💥 (plantage).
# - chrono(fonction, ...) mesure le temps sur une grande entrée (tests de vitesse facultatifs).
# Tu n'as pas besoin de comprendre ce code pour t'en servir.

import copy
import time

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas, ne_pas_modifier=False):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        copie = copy.deepcopy(arguments)
        try:
            obtenu = fonction(*copie)
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu != attendu:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
        elif ne_pas_modifier and copie != arguments:
            print(f"⚠️ {fonction.__name__}({texte_args}) -> bon résultat, mais ta fonction a modifié son entrée (interdit ici)")
        else:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
    print(f"\n{reussis}/{len(cas)} tests réussis")

def chrono(fonction, *arguments, attendu=None, limite=1.0):
    debut = time.perf_counter()
    try:
        obtenu = fonction(*arguments)
    except Exception as erreur:
        print(f"💥 {fonction.__name__} a planté : {type(erreur).__name__}: {erreur}")
        return
    duree = time.perf_counter() - debut
    if obtenu is None:
        print("⏳ Ta fonction ne renvoie rien pour l'instant : écris-la d'abord.")
    elif attendu is not None and obtenu != attendu:
        print(f"❌ Résultat faux sur la grande entrée : {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    elif duree <= limite:
        print(f"✅ {duree:.3f} s : assez rapide")
    else:
        print(f"⚠️ {duree:.1f} s : trop lent (limite : {limite} s). Ta solution n'a probablement pas la complexité attendue.")

## Exercice 36 — Compter les multiples sans boucle

**Idée** : entre 1 et `x`, il y a exactement `x // k` multiples de `k` (k, 2k, 3k…). Le nombre de multiples dans `[debut, fin]` est donc « ceux jusqu'à `fin` » moins « ceux jusqu'à `debut - 1` ». La division entière de Python arrondit vers le bas, ce qui donne aussi le bon résultat quand `debut = 0` : `(0 - 1) // k = -1`, et le 0 est bien compté.

**Complexité** : O(1).

In [ ]:
def nombre_multiples(debut: int, fin: int, k: int) -> int:
    # x // k = nombre de multiples de k dans 1, 2, ..., x   (et -1 pour x = -1)
    # multiples dans [debut, fin] = (multiples jusqu'à fin) − (multiples jusqu'à debut − 1)
    return fin // k - (debut - 1) // k


verifier(nombre_multiples, [
    ((4, 20, 3), 5),
    ((0, 0, 5), 1),   # 0 est divisible par tout k
    ((1, 1, 2), 0),
    ((6, 6, 3), 1),
    ((0, 10, 1), 11),
    ((10, 19, 10), 1),
    ((7, 14, 7), 2),   # les deux bornes sont des multiples
])

In [ ]:
# Test de vitesse
chrono(nombre_multiples, 1, 10**12, 7, attendu=142_857_142_857)

## Exercice 37 — Réduire une fraction

**Idée** : diviser `a` et `b` par leur **PGCD** (plus grand commun diviseur), calculé avec l'algorithme d'Euclide : `pgcd(x, y) = pgcd(y, x % y)`, jusqu'à ce que `y` vaille 0. Puis normaliser le signe pour que le dénominateur soit positif.

**Complexité** : O(log(min(|a|, |b|))).

In [ ]:
def fraction_reduite(a: int, b: int) -> tuple[int, int]:
    def pgcd(x, y):                  # algorithme d'Euclide (math.gcd fait la même chose)
        while y:
            x, y = y, x % y
        return abs(x)

    if a == 0:
        return (0, 1)
    d = pgcd(a, b)
    a, b = a // d, b // d
    if b < 0:                        # le signe va au numérateur
        a, b = -a, -b
    return (a, b)


verifier(fraction_reduite, [
    ((8, 12), (2, 3)),
    ((4, -6), (-2, 3)),
    ((0, 7), (0, 1)),
    ((0, -3), (0, 1)),
    ((-5, -10), (1, 2)),   # deux négatifs : résultat positif
    ((7, 1), (7, 1)),
    ((9, 3), (3, 1)),
    ((-12, 18), (-2, 3)),
])

## Exercice 38 — Nombre narcissique

**Idée** : passer par la chaîne `str(n)` donne à la fois le nombre de chiffres (`len`) et chaque chiffre. Version purement arithmétique : extraire les chiffres avec `n % 10` et `n // 10`.

**Complexité** : O(d), d = nombre de chiffres.

In [ ]:
def est_armstrong(n: int) -> bool:
    chiffres = str(n)
    d = len(chiffres)                 # str(0) vaut "0" : 1 chiffre
    return sum(int(c) ** d for c in chiffres) == n


verifier(est_armstrong, [
    ((153,), True),
    ((123,), False),
    ((0,), True),
    ((7,), True),   # un chiffre : 7¹ = 7
    ((10,), False),
    ((9474,), True),
    ((9475,), False),
    ((370,), True),
])

## Exercice 39 — Rendre la monnaie

**Idée** : un algorithme **glouton** : prendre à chaque fois la plus grande pièce possible. Avec `//` et `%`, on prend d'un coup toutes les pièces d'une valeur.

Attention : le glouton est optimal **pour ce système de pièces**, pas pour tous. Avec des pièces {1, 3, 4} et un montant de 6, il donne 4 + 1 + 1 (3 pièces) alors que 3 + 3 (2 pièces) est meilleur. Dans ce cas, il faut la programmation dynamique (thème 8).

**Complexité** : O(nombre de valeurs de pièces).

In [ ]:
def minimum_pieces(montant: int) -> int:
    nombre = 0
    for piece in [50, 20, 10, 5, 2, 1]:      # de la plus grande à la plus petite
        nombre += montant // piece            # autant de cette pièce que possible
        montant %= piece                      # ce qu'il reste à rendre
    return nombre


verifier(minimum_pieces, [
    ((87,), 5),
    ((0,), 0),
    ((1,), 1),
    ((4,), 2),
    ((99,), 6),
    ((100,), 2),
    ((38,), 5),
    ((1000,), 20),
])

## Exercice 40 — Choisir un maximum de réunions

**Idée** : le glouton classique : trier par **heure de fin**, puis prendre chaque réunion qui commence après la fin de la dernière réunion choisie. Choisir la réunion qui se termine le plus tôt laisse le plus de temps libre pour les suivantes. Trier par début ou par durée ne marche pas.

**Complexité** : O(n log n), à cause du tri.

In [ ]:
def max_reunions(plages: list[list[int]]) -> int:
    nombre = 0
    fin_derniere = None
    for debut, fin in sorted(plages, key=lambda p: p[1]):    # celle qui finit le plus tôt d'abord
        if fin_derniere is None or debut >= fin_derniere:     # compatible avec la dernière choisie
            nombre += 1
            fin_derniere = fin
    return nombre


verifier(max_reunions, [
    (([[0, 3], [1, 2], [2, 4], [3, 5], [5, 6]],), 3),
    (([],), 0),
    (([[1, 2]],), 1),
    (([[1, 5], [2, 3], [3, 4]],), 2),
    (([[1, 2], [1, 2], [1, 2]],), 1),   # trois fois la même réunion
    (([[1, 2], [2, 3], [3, 4]],), 3),   # elles s'enchaînent exactement
    (([[0, 10], [1, 2], [2, 3], [3, 4], [4, 5]],), 4),
])

## Exercice 41 — Terminer le plus de petites tâches possible

**Idée** : glouton : pour faire le **plus grand nombre** de tâches, on commence par les plus courtes. Dès qu'une tâche ne rentre plus dans le budget, les suivantes (plus longues) ne rentrent pas non plus.

**Complexité** : O(n log n), à cause du tri.

In [ ]:
def maximum_taches(durees: list[int], budget: int) -> int:
    nombre = 0
    for d in sorted(durees):        # les plus courtes d'abord
        if d > budget:
            break                   # les suivantes sont encore plus longues
        budget -= d
        nombre += 1
    return nombre


verifier(maximum_taches, [
    (([6, 1, 2, 3], 6), 3),
    (([], 10), 0),
    (([5], 4), 0),
    (([5], 5), 1),   # exactement le budget : ça passe
    (([1, 1, 1, 1], 2), 2),
    (([4, 2, 8, 1], 7), 3),
    (([10, 20, 30], 100), 3),
])

---

[Énoncés de la série](../serie7_maths_gloutons.ipynb)